# 🧪 Lab 6: The Engine Room Firefight (Executors Tab Imbalance & GC Diagnostics)

In this laboratory session, we audit Spark's worker metrics to observe executor-level symptoms, task imbalance, GC signals when they appear, and live thread state snapshots.

**Objective:** Identify task-level imbalance under data skew, inspect executor metrics, and look for JVM memory-pressure signals when the workload produces them.


### Step 1: Initialize the Fleet Monitoring Station
We start a local Spark session, enable the Spark Web UI, and verify the actual UI URL.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import time

existing = SparkSession.getActiveSession()
if existing is not None:
    existing.stop()

spark = (
    SparkSession.builder
    .appName("spark-ui-executors-engine-room-lab-06")
    .master("local[*]")
    .config("spark.ui.enabled", "true")
    .config("spark.ui.port", "4040")
    .config("spark.port.maxRetries", "16")
    .config("spark.ui.threadDumpsEnabled", "true")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "32")
    .getOrCreate()
)

print(f"Active Spark version: {spark.version}")
print(f"🛰️ Actual Spark UI: {spark.sparkContext.uiWebUrl}")


Active Spark version: 4.1.2
🛰️ Actual Spark UI: http://T14-PF4WM3XL.sdggroup.local:4040


### Step 2: Inject Severe Key Asymmetry (Task Imbalance Pass)
We fabricate a heavy, highly unbalanced dataset where the vast majority of rows share an identical routing identifier. This is designed to create one very large key, which may produce a much heavier shuffle partition later.


In [2]:
# Generate an intense data skew profile to create uneven task work later.
asymmetric_stream = (
    spark.range(0, 8_000_000, 1, numPartitions=64)
    .withColumn(
        "sector_id",
        F.when(F.col("id") < 7_600_000, F.lit("ALPHA_QUADRANT"))
         .otherwise(F.concat(F.lit("sector_"), (F.col("id") % 10_000).cast("string")))
    )
)

print("🛡️ Asymmetric workload blueprint mapped. Gravitational data skew staged.")
print("No action has run yet, so no distributed work has been executed from this DataFrame.")


🛡️ Asymmetric workload blueprint mapped. Gravitational data skew staged.
No action has run yet, so no distributed work has been executed from this DataFrame.


### Step 3: Trigger High-Friction Execution (The Task Runtime Burn)
We execute a shuffle-heavy repartition and partition sort with a generated checksum payload. This may expose skew symptoms in task duration, shuffle read size, GC time, or executor metrics.

In local mode, do not expect a clean multi-executor imbalance story. The Tasks table is usually the better evidence source.


In [3]:
import tempfile

spark.sparkContext.setJobDescription("🔥 Engine Test: Skewed Repartition and Executor Metrics")

skewed_payload = asymmetric_stream.withColumn(
    "checksum",
    F.sha2(F.col("id").cast("string"), 256)
)

output_path = tempfile.mkdtemp(prefix="spark_ui_lab06_engine_room_")

start_clock = time.time()

(
    skewed_payload
    .repartition(32, "sector_id")
    .sortWithinPartitions("checksum")
    .write
    .mode("overwrite")
    .parquet(output_path)
)

end_clock = time.time()

print(f"🎯 Skewed shuffle/write completed in {end_clock - start_clock:.2f} seconds.")
print(f"Temporary output path: {output_path}")
print("Inspect Stages → Tasks first, then Executors for GC time, task time, logs, and thread dump links.")


🎯 Skewed shuffle/write completed in 27.68 seconds.
Temporary output path: C:\Users\ANGEL~1.ALV\AppData\Local\Temp\spark_ui_lab06_engine_room_2f6v5xm2
Inspect Stages → Tasks first, then Executors for GC time, task time, logs, and thread dump links.


### Step 4: Inspect Thread State and Keep the Live UI Open
We keep the driver process alive so the Live UI remains available long enough to inspect executor metrics, logs, and optional thread dumps.


In [4]:
import shutil

print(f"⏳ Explore the Executors tab now: {spark.sparkContext.uiWebUrl}")
print("👉 Click the Thread Dump link if your UI/platform exposes it.")
input("Press Enter when you are done inspecting the Executors tab and want to stop Spark...")

if "output_path" in globals():
    shutil.rmtree(output_path, ignore_errors=True)
    print("🧹 Temporary output cleaned up.")

spark.stop()
print("💀 SparkContext stopped. The Live UI is no longer available.")


⏳ Explore the Executors tab now: http://T14-PF4WM3XL.sdggroup.local:4040
👉 Click the Thread Dump link if your UI/platform exposes it.
🧹 Temporary output cleaned up.
💀 SparkContext stopped. The Live UI is no longer available.


## 📊 Post-Lab Analysis: Evidence from the Engine Room

### 1. Executor Metrics Are Physical Symptoms

The Executors tab helps attach Spark symptoms to executor IDs and hosts: task time, failed tasks, shuffle metrics, storage memory, disk usage, logs, and GC time.

In local mode, executor-level imbalance may be limited or invisible because the application may have only one executor. For skew, inspect the Stages tab and sort the Tasks table by Duration, Shuffle Read Size, Spill, and GC Time.

### 2. GC Is a Signal, Not a Guaranteed Outcome

This lab may show elevated JVM GC time if the workload creates enough memory pressure. The exact result depends on Spark version, available memory, local task slots, and machine load.

Do not claim GC pressure just because a workload uses hashing or repartitioning. Verify it in the Executors tab and task metrics.

### 3. Thread Dumps Are Diagnostic Snapshots

Thread dumps show what executor JVM threads are doing at the moment you capture them. They can help identify blocked I/O, locks, long-running tasks, or idle executor threads.

They do not automatically identify the root cause. Use them together with task metrics, executor logs, and the cluster manager.

### ⏱️ Validation Summary

* **Step 1:** Started a local Spark session and printed the actual Spark UI URL.
* **Step 2:** Created an intentionally skewed dataset without triggering execution yet.
* **Step 3:** Triggered a shuffle-heavy repartition, sort, and Parquet write to inspect task/executor symptoms.
* **Step 4:** Kept the Live UI open for executor metrics, logs, and optional thread dump inspection, then cleaned up the temporary output.
